<a href="https://colab.research.google.com/github/VaibhavRajput022/Codsoft/blob/main/MOVIE_RATING_PREDICTION_WITH_PYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install scikit-surprise to easily load MovieLens dataset
%pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.8 MB/s eta 0:00:00


In [4]:
import pandas as pd
from surprise import Dataset, Reader

# Define the format of the dataset
reader = Reader(line_format='user item rating timestamp', sep='\t')

# Load the MovieLens 100k dataset. This will download it if not already present.
data = Dataset.load_builtin('ml-100k', reader)

# Convert the dataset to a pandas DataFrame for easier manipulation and exploration
# This involves iterating through the raw data
raw_ratings = data.raw_ratings
df_ratings = pd.DataFrame(raw_ratings, columns=['user_id', 'movie_id', 'rating', 'timestamp'])

# Convert 'rating' to numeric type
df_ratings['rating'] = pd.to_numeric(df_ratings['rating'])

# Display the first 5 rows of the ratings DataFrame
display(df_ratings.head())

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


,user_id,movie_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


The MovieLens dataset often comes with separate files for movie information (title, genres, etc.). Let's also load the movie information to get features like genre and title.

In [5]:
import os
import zipfile
import requests

# Path where MovieLens dataset is usually stored by surprise library
dataset_path = os.path.expanduser('~') + '/.surprise_data/ml-100k'

# The actual u.item file is inside a folder named 'ml-100k' inside the dataset_path
movies_file_path = os.path.join(dataset_path, 'ml-100k', 'u.item')

# If the directory doesn't exist, it means the dataset wasn't downloaded by surprise
# We'll try to download it directly if it's missing.
if not os.path.exists(movies_file_path):
    print("MovieLens 100k dataset not found by surprise. Attempting manual download...")
    url = 'http://files.grouplens.org/datasets/movielens/ml-100k.zip'
    zip_path = os.path.join(os.path.expanduser('~'), 'ml-100k.zip')

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(os.path.dirname(movies_file_path)) # Extract to parent dir of movies_file_path
        print("MovieLens 100k dataset downloaded and extracted successfully.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading or extracting dataset: {e}")
        print("Please ensure you have an active internet connection or try again later.")
        # Exit or raise error if dataset can't be loaded
        raise

# Load movie data
# The u.item file has no header and uses '|' as a separator
# The columns are: movie id | movie title | release date | video release date | IMDb URL |
#                  unknown | Action | Adventure | Animation | Children's | Comedy | Crime |
#                  Documentary | Drama | Fantasy | Film-Noir | Horror | Musical | Mystery |
#                  Romance | Sci-Fi | Thriller | War | Western

movies_df = pd.read_csv(
    movies_file_path,
    sep='|',
    header=None,
    encoding='latin-1', # or 'ISO-8859-1'
    names=[
        'movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
        'unknown', 'Action', 'Adventure', 'Animation', 'Children',
        'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
        'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller',
        'War', 'Western'
    ]
)

# Select relevant columns from movie data
movies_df = movies_df[['movie_id', 'title', 'release_date', 'Action', 'Adventure', 'Animation', 'Children',
                     'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
                     'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller',
                     'War', 'Western']]

# Display the first 5 rows of the movies DataFrame
display(movies_df.head())

,movie_id,title,release_date,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [10]:
import pandas as pd

# Check if df_merged is already defined, otherwise merge DataFrames
# This is to handle cases where previous cells might not have been executed or kernel state was reset.
if 'df_merged' not in locals() and 'df_merged' not in globals():
    # Ensure movie_id columns have the same type before merging
    # Convert 'movie_id' in df_ratings to int to match movies_df
    if 'df_ratings' in locals() and 'movies_df' in locals():
        df_ratings['movie_id'] = df_ratings['movie_id'].astype(int)
        # Merge the ratings and movie information DataFrames
        df_merged = pd.merge(df_ratings, movies_df, on='movie_id', how='left')
        # Convert release_date to datetime objects
        df_merged['release_date'] = pd.to_datetime(df_merged['release_date'])
    else:
        print("Error: df_ratings or movies_df not found. Please ensure previous cells are executed.")
        # You might want to raise an exception or handle this more robustly
        raise NameError("df_ratings or movies_df not found")

# Get basic information about the merged DataFrame
print(df_merged.info())

# Check for missing values
print("\nMissing values:")
display(df_merged.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   user_id       100000 non-null  object        
 1   movie_id      100000 non-null  int64         
 2   rating        100000 non-null  float64       
 3   timestamp     100000 non-null  object        
 4   title         100000 non-null  object        
 5   release_date  99991 non-null   datetime64[ns]
 6   Action        100000 non-null  int64         
 7   Adventure     100000 non-null  int64         
 8   Animation     100000 non-null  int64         
 9   Children      100000 non-null  int64         
 10  Comedy        100000 non-null  int64         
 11  Crime         100000 non-null  int64         
 12  Documentary   100000 non-null  int64         
 13  Drama         100000 non-null  int64         
 14  Fantasy       100000 non-null  int64         
 15  Film-Noir     1000

,0
user_id,0
movie_id,0
rating,0
timestamp,0
title,0
release_date,9
Action,0
Adventure,0
Animation,0
Children,0


### Step 2: Data Preprocessing

Based on the initial exploration, we need to perform the following preprocessing steps:

1.  **Handle Missing Values**: Drop rows with missing `release_date` values.
2.  **Convert Data Types**: Convert `timestamp` and `user_id` columns to appropriate data types.

In [11]:
# 1. Handle Missing Values: Drop rows with missing 'release_date'
original_rows = df_merged.shape[0]
df_merged.dropna(subset=['release_date'], inplace=True)
print(f"Dropped {original_rows - df_merged.shape[0]} rows with missing release_date.")

# 2. Convert Data Types
# Convert 'timestamp' to datetime
df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'], unit='s')

# Convert 'user_id' from object to int64
df_merged['user_id'] = df_merged['user_id'].astype(int)

# Verify changes
print("\nDataFrame info after preprocessing:")
df_merged.info()

print("\nMissing values after preprocessing:")
display(df_merged.isnull().sum())

Dropped 9 rows with missing release_date.


/tmp/ipykernel_3228/2286916737.py:8: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  df_merged['timestamp'] = pd.to_datetime(df_merged['timestamp'], unit='s')



DataFrame info after preprocessing:
<class 'pandas.core.frame.DataFrame'>
Index: 99991 entries, 0 to 99999
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   user_id       99991 non-null  int64         
 1   movie_id      99991 non-null  int64         
 2   rating        99991 non-null  float64       
 3   timestamp     99991 non-null  datetime64[ns]
 4   title         99991 non-null  object        
 5   release_date  99991 non-null  datetime64[ns]
 6   Action        99991 non-null  int64         
 7   Adventure     99991 non-null  int64         
 8   Animation     99991 non-null  int64         
 9   Children      99991 non-null  int64         
 10  Comedy        99991 non-null  int64         
 11  Crime         99991 non-null  int64         
 12  Documentary   99991 non-null  int64         
 13  Drama         99991 non-null  int64         
 14  Fantasy       99991 non-null  int64         
 15  Film

,0
user_id,0
movie_id,0
rating,0
timestamp,0
title,0
release_date,0
Action,0
Adventure,0
Animation,0
Children,0


### Step 3: Feature Engineering

We will create new features from existing ones that could provide more predictive power to our model.

1.  **From `release_date`**: Extract features like `release_year`, `release_month`, and `release_day_of_week`.
2.  **From `timestamp`**: Extract features like `rating_year`, `rating_month`, and `rating_day_of_week`.

In [12]:
# 1. Features from release_date
df_merged['release_year'] = df_merged['release_date'].dt.year
df_merged['release_month'] = df_merged['release_date'].dt.month
df_merged['release_day_of_week'] = df_merged['release_date'].dt.dayofweek # Monday=0, Sunday=6

# 2. Features from timestamp (when the rating was given)
df_merged['rating_year'] = df_merged['timestamp'].dt.year
df_merged['rating_month'] = df_merged['timestamp'].dt.month
df_merged['rating_day_of_week'] = df_merged['timestamp'].dt.dayofweek

# Display the first few rows with new features
display(df_merged[['release_date', 'release_year', 'release_month', 'release_day_of_week',
                     'timestamp', 'rating_year', 'rating_month', 'rating_day_of_week']].head())

,release_date,release_year,release_month,release_day_of_week,timestamp,rating_year,rating_month,rating_day_of_week
0,1997-01-24,1997,1,4,1997-12-04 15:55:49,1997,12,3
1,1997-01-01,1997,1,2,1998-04-04 19:22:22,1998,4,5
2,1994-01-01,1994,1,5,1997-11-07 07:18:36,1997,11,4
3,1994-01-01,1994,1,5,1997-11-27 05:02:03,1997,11,3
4,1997-01-01,1997,1,2,1998-02-02 05:33:16,1998,2,0


### Step 4: Model Building - Data Preparation

Before building the model, we need to split our dataset into features (X) and the target variable (y). The target variable `y` will be the `rating`.

For features `X`, we will include the genre columns, and the newly engineered date features. We will drop `user_id`, `movie_id`, `title`, `release_date`, and `timestamp` as these are either identifiers or have been transformed into other features.

In [14]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
# Drop original identifiers and dates which have been engineered into new features
X = df_merged.drop(columns=['user_id', 'movie_id', 'rating', 'title', 'release_date', 'timestamp'])
y = df_merged['rating']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split the data into training and testing sets
# We'll use a test size of 20% and a random state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Display the first few rows of the training features
display(X_train.head())

Features shape: (99991, 24)
Target shape: (99991,)
X_train shape: (79992, 24)
X_test shape: (19999, 24)
y_train shape: (79992,)
y_test shape: (19999,)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,...,Sci-Fi,Thriller,War,Western,release_year,release_month,release_day_of_week,rating_year,rating_month,rating_day_of_week
38244,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,1951,1,0,1997,11,3
3808,1,0,0,0,0,0,0,0,0,0,...,1,1,1,0,1986,1,2,1997,12,1
27931,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,1995,1,6,1997,11,5
6008,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1997,1,4,1997,9,6
65811,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,1967,1,6,1997,11,4


### Step 4: Model Building - Training a Linear Regression Model

We will now train a `LinearRegression` model using our prepared training data (`X_train` and `y_train`).

In [15]:
from sklearn.linear_model import LinearRegression

# Initialize the Linear Regression model
linear_model = LinearRegression()

# Train the model
linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


### Step 5: Model Evaluation

Now, let's evaluate the performance of our trained `LinearRegression` model using the test set (`X_test` and `y_test`). We will use the following metrics:

*   **Mean Absolute Error (MAE)**: The average of the absolute differences between predictions and actual values.
*   **Mean Squared Error (MSE)**: The average of the squared differences between predictions and actual values. This penalizes larger errors more heavily.
*   **R-squared (R2)**: Represents the proportion of the variance in the dependent variable that is predictable from the independent variables.

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make predictions on the test set
y_pred = linear_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5 # Root Mean Squared Error
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared (R2): {r2:.4f}")

Mean Absolute Error (MAE): 0.8948
Mean Squared Error (MSE): 1.1868
Root Mean Squared Error (RMSE): 1.0894
R-squared (R2): 0.0634
